In [0]:
%pip install -U -qqqq mlflow langgraph==0.3.4 databricks-langchain databricks-agents uv  databricks-vectorsearch --upgrade langgraph
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ./00_init_requirements

Catalog created demo_rag
Schema created demo_rag.workday_demos
Volume created /Volumes/demo_rag/workday_demos/workday_unstructure_data


# Create Vector Search Indexes

Create the indexes via python SDK. There are two steps involved:

1. Create vector search endpoint (one endpoint can serve multiple vector search indexes)
2. Create vector search indexes for different data types:
   - Email communications
   - Meeting notes
   - Customer feedback
   - Employee records
   - Job requisitions

In [0]:
from pyspark.sql.functions import expr, col, explode, monotonically_increasing_id

# Parse PDF documents from volumes using ai_parse_document
customer_feedback_parsed = (
    spark.read.format("binaryFile")
    .load(f"/Volumes/{catalog_name}/{schema_name}/workday_unstructure_data/customer_feedback/")
    .withColumn("parsed", expr("ai_parse_document(content, map('version', '2.0'))"))
    .withColumn("content", expr("array_join(transform(parsed:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n')"))
    .select(
        "content",
        expr("parsed:document").alias("document"),
        expr("parsed:document:pages").alias("pages"),
        expr("parsed:error_status").alias("error_status"),
        col("path").alias("doc_uri")
    )
)

meeting_notes_parsed = (
    spark.read.format("binaryFile")
    .load(f"/Volumes/{catalog_name}/{schema_name}/workday_unstructure_data/meeting_notes/")
    .withColumn("parsed", expr("ai_parse_document(content, map('version', '2.0'))"))
    .withColumn("content", expr("array_join(transform(parsed:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n')"))
    .select(
        "content",
        expr("parsed:document").alias("document"),
        expr("parsed:document:pages").alias("pages"),
        expr("parsed:error_status").alias("error_status"),
        col("path").alias("doc_uri")
    )
)

email_communications_parsed = (
    spark.read.format("binaryFile")
    .load(f"/Volumes/{catalog_name}/{schema_name}/workday_unstructure_data/email_communications/")
    .withColumn("parsed", expr("ai_parse_document(content, map('version', '2.0'))"))
    .withColumn("content", expr("array_join(transform(parsed:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n')"))
    .select(
        "content",
        expr("parsed:document").alias("document"),
        expr("parsed:document:pages").alias("pages"),
        expr("parsed:error_status").alias("error_status"),
        col("path").alias("doc_uri")
    )
)

In [0]:
%sql
 SELECT transform(array(1, 2, 3), x -> x + 1);
 [2,3,4]
> SELECT transform(array(1, 2, 3), (x, i) -> x + i);
 [1,3,5]

In [0]:
# Load binary PDF files from a volume
pdf_df = spark.read.format("binaryFile").load(f"/Volumes/{catalog_name}/{schema_name}/workday_unstructure_data/customer_feedback/")

display(pdf_df)

# Parse documents using the Databricks AI function ai_parse_document
parsed_df = pdf_df.withColumn(
    "parsed",
    expr("ai_parse_document(content, map('version', '2.0'))")
)

display(parsed_df)
# The code below takes each PDF's parsed content and creates a single 'content' column by joining all extracted text elements.
# It then selects additional columns for document structure, page info, error status, and file path.

extracted_df = parsed_df.withColumn(
    "content",
    expr("array_join(transform(parsed:document.elements::ARRAY<STRUCT<content:STRING>>, x -> x.content), '\n')")
).select(
    "content",
    expr("parsed:document"),
    expr("parsed:document:pages"),
    expr("parsed:error_status"),
    col("path").alias("doc_uri")
)

display(extracted_df)

path,modificationTime,length,content
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00004.pdf,2026-08-30T14:32:11.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgOCAwIFIgL01lZGlhQm94IFsgMCAwIDYxMiA3OTIgXSAvUGFyZW50IDcgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjUgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyA3IDAgUiAvVHlwZSAvQ2F0YWxvZwo+PgplbmRvYmoKNiAwIG9iago8PAovQXV0aG9yIChhbm9ueW1vdXMpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvQ3JlYXRvciAoYW5vbnltb3VzKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAodW5zcGVjaWZpZWQpIC9UaXRsZSAodW50aXRsZWQpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKNyAwIG9iago8PAovQ291bnQgMSAvS2lkcyBbIDQgMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDQ0NAo+PgpzdHJlYW0KR2FzMkVkO0ZPaSdTYygpTVIwcDNSQi4pUmthZUdbOT1BTXUoWzAjKFlzRS50MSVYZDYjV1YiJTFjM0o3SWZGbkRsMm42LkQuWldGYltQZyVdJjZKQkdNcj1YJitMI1lpXidLKzddNUUhW0hBSG4zVDYlZVtgNmdjTjVlJGE/PVIkT0QsY2NpJ3JZUHRSKFJHLyNvaFlNRD0vUEdKR0IrWT4zP2YjQCxuSUs0U18rYy1hOSpRcXRcIk9jRVlgM1opSjs3UCw9cU07KyhadDhPKDUrWysybENbWlslPTJYIm5oLUgnS0ksIkxMJFYvcE5uIWdITT4saU9naHJOK1c0REdjayhZZ1dhK1dtIi5bXmpfL2VRUD03aWRQZDpLIT1TY2ombWk8IzdjYkNpbTg3cWcuOVhNPFFUUl5tbFMwPE1fSCFGZEt0IT0yaTM9QDBnPGZrRGZxN1wkOHQqPSdySmw+Tk5RJTciNEtuNEdAJWhKci44L1E+cWpTM1IuSW50KSI+b1tzPUVobGlsRjUjS3E+X0wnTmExOjojNSp1YVskIilfSCRPNTVadShTPjxXaWlGIUlDbSQ1bH4+ZW5kc3RyZWFtCmVuZG9iagp4cmVmCjAgOQowMDAwMDAwMDAwIDY1NTM1IGYgCjAwMDAwMDAwNjEgMDAwMDAgbiAKMDAwMDAwMDEwMiAwMDAwMCBuIAowMDAwMDAwMjA5IDAwMDAwIG4gCjAwMDAwMDAzMjEgMDAwMDAgbiAKMDAwMDAwMDUxNCAwMDAwMCBuIAowMDAwMDAwNTgyIDAwMDAwIG4gCjAwMDAwMDA4NDMgMDAwMDAgbiAKMDAwMDAwMDkwMiAwMDAwMCBuIAp0cmFpbGVyCjw8Ci9JRCAKWzwyNmQ3MDlmZmRkNWUwYjBmYTZhNWUzMjE0ZDBkNDVmNz48MjZkNzA5ZmZkZDVlMGIwZmE2YTVlMzIxNGQwZDQ1Zjc+XQolIFJlcG9ydExhYiBnZW5lcmF0ZWQgUERGIGRvY3VtZW50IC0tIGRpZ2VzdCAob3BlbnNvdXJjZSkKCi9JbmZvIDYgMCBSCi9Sb290IDUgMCBSCi9TaXplIDkKPj4Kc3RhcnR4cmVmCjE0MzYKJSVFT0YK
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00009.pdf,2026-08-30T14:32:12.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgOCAwIFIgL01lZGlhQm94IFsgMCAwIDYxMiA3OTIgXSAvUGFyZW50IDcgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjUgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyA3IDAgUiAvVHlwZSAvQ2F0YWxvZwo+PgplbmRvYmoKNiAwIG9iago8PAovQXV0aG9yIChhbm9ueW1vdXMpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA4MzAxNDMyMTIrMDAnMDAnKSAvQ3JlYXRvciAoYW5vbnltb3VzKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA4MzAxNDMyMTIrMDAnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAodW5zcGVjaWZpZWQpIC9UaXRsZSAodW50aXRsZWQpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKNyAwIG9iago8PAovQ291bnQgMSAvS2lkcyBbIDQgMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoID

path,modificationTime,length,content,parsed
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00004.pdf,2026-08-30T14:32:11.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgOCAwIFIgL01lZGlhQm94IFsgMCAwIDYxMiA3OTIgXSAvUGFyZW50IDcgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjUgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyA3IDAgUiAvVHlwZSAvQ2F0YWxvZwo+PgplbmRvYmoKNiAwIG9iago8PAovQXV0aG9yIChhbm9ueW1vdXMpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvQ3JlYXRvciAoYW5vbnltb3VzKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAodW5zcGVjaWZpZWQpIC9UaXRsZSAodW50aXRsZWQpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKNyAwIG9iago8PAovQ291bnQgMSAvS2lkcyBbIDQgMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDQ0NAo+PgpzdHJlYW0KR2FzMkVkO0ZPaSdTYygpTVIwcDNSQi4pUmthZUdbOT1BTXUoWzAjKFlzRS50MSVYZDYjV1YiJTFjM0o3SWZGbkRsMm42LkQuWldGYltQZyVdJjZKQkdNcj1YJitMI1lpXidLKzddNUUhW0hBSG4zVDYlZVtgNmdjTjVlJGE/PVIkT0QsY2NpJ3JZUHRSKFJHLyNvaFlNRD0vUEdKR0IrWT4zP2YjQCxuSUs0U18rYy1hOSpRcXRcIk9jRVlgM1opSjs3UCw9cU07KyhadDhPKDUrWysybENbWlslPTJYIm5oLUgnS0ksIkxMJFYvcE5uIWdITT4saU9naHJOK1c0REdjayhZZ1dhK1dtIi5bXmpfL2VRUD03aWRQZDpLIT1TY2ombWk8IzdjYkNpbTg3cWcuOVhNPFFUUl5tbFMwPE1fSCFGZEt0IT0yaTM9QDBnPGZrRGZxN1wkOHQqPSdySmw+Tk5RJTciNEtuNEdAJWhKci44L1E+cWpTM1IuSW50KSI+b1tzPUVobGlsRjUjS3E+X0wnTmExOjojNSp1YVskIilfSCRPNTVadShTPjxXaWlGIUlDbSQ1bH4+ZW5kc3RyZWFtCmVuZG9iagp4cmVmCjAgOQowMDAwMDAwMDAwIDY1NTM1IGYgCjAwMDAwMDAwNjEgMDAwMDAgbiAKMDAwMDAwMDEwMiAwMDAwMCBuIAowMDAwMDAwMjA5IDAwMDAwIG4gCjAwMDAwMDAzMjEgMDAwMDAgbiAKMDAwMDAwMDUxNCAwMDAwMCBuIAowMDAwMDAwNTgyIDAwMDAwIG4gCjAwMDAwMDA4NDMgMDAwMDAgbiAKMDAwMDAwMDkwMiAwMDAwMCBuIAp0cmFpbGVyCjw8Ci9JRCAKWzwyNmQ3MDlmZmRkNWUwYjBmYTZhNWUzMjE0ZDBkNDVmNz48MjZkNzA5ZmZkZDVlMGIwZmE2YTVlMzIxNGQwZDQ1Zjc+XQolIFJlcG9ydExhYiBnZW5lcmF0ZWQgUERGIGRvY3VtZW50IC0tIGRpZ2VzdCAob3BlbnNvdXJjZSkKCi9JbmZvIDYgMCBSCi9Sb290IDUgMCBSCi9TaXplIDkKPj4Kc3RhcnR4cmVmCjE0MzYKJSVFT0YK,"{""document"":{""elements"":[{""bbox"":[{""coord"":[593,125,1108,169],""page_id"":0}],""confidence"":1,""content"":""Customer Feedback Report"",""description"":null,""id"":0,""type"":""title""},{""bbox"":[{""coord"":[197,235,1379,686],""page_id"":0}],""confidence"":1,""content"":""Feedback ID: FB00004\nAccount ID: ACC00680\nOpportunity ID: OPP001604\nFeedback Date: 2026-08-27\nSentiment: Positive\nSentiment Score: 0.768\nCustomer Role: Operations Manager\nSource: Survey\n\nFeedback Content:\nExcellent demo! Highlights included real-time reporting, a clean interface, and easy integrations."",""description"":null,""id"":1,""type"":""text""}],""pages"":[{""id"":0,""image_uri"":null}]},""error_status"":null,""metadata"":{""file_metadata"":null,""id"":""55ed778e-f5a2-4ba2-bf7d-1028c54037e8"",""version"":""2.0""}}"
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00009.pdf,2026-08-30T14:32:12.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVud

content,document,pages,error_status,doc_uri
Customer Feedback Report Feedback ID: FB00009 Account ID: ACC00370 Opportunity ID: OPP001437 Feedback Date: 2026-07-20 Sentiment: Positive Sentiment Score: 0.624 Customer Role: CTO Source: Phone Interview Feedback Content: Outstanding experience with the Workday sales team Clear communication and excellent support.,"{""elements"":[{""bbox"":[{""coord"":[592,125,1108,169],""page_id"":0}],""confidence"":1,""content"":""Customer Feedback Report"",""description"":null,""id"":0,""type"":""title""},{""bbox"":[{""coord"":[197,235,860,726],""page_id"":0}],""confidence"":1,""content"":""Feedback ID: FB00009\nAccount ID: ACC00370\nOpportunity ID: OPP001437\nFeedback Date: 2026-07-20\nSentiment: Positive\nSentiment Score: 0.624\nCustomer Role: CTO\nSource: Phone Interview\n\nFeedback Content:\nOutstanding experience with the Workday sales team\nClear communication and excellent support."",""description"":null,""id"":1,""type"":""text""}],""pages"":[{""id"":0,""image_uri"":null}]}","[{""id"":0,""image_uri"":null}]",null,dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00009.pdf
"Customer Feedback Report Feedback ID: FB00004 Account ID: ACC00680 Opportunity ID: OPP001604 Feedback Date: 2026-08-27 Sentiment: Positive Sentiment Score: 0.768 Customer Role: Operations Manager Source: Survey Feedback Content: Excellent demo! Highlights included real-time reporting, a clean interface, and easy integrations.","{""elements"":[{""bbox"":[{""coord"":[593,125,1108,169],""page_id"":0}],""confidence"":1,""content"":""Customer Feedback Report"",""description"":null,""id"":0,""type"":""title""},{""bbox"":[{""coord"":[197,235,1379,686],""page_id"":0}],""confidence"":1,""content"":""Feedback ID: FB00004\nAccount ID: ACC00680\nOpportunity ID: OPP001604\nFeedback Date: 2026-08-27\nSentiment: Positive\nSentiment Score: 0.768\nCustomer Role: Operations Manager\nSource: Survey\n\nFeedback Content:\nExcellent demo! Highlights included real-time reporting, a clean interface, and easy integrations."",""description"":null,""id"":1,""type"":""text""}],""pages"":[{""id"":0,""image_uri"":null}]}","[{""id"":0,""image_uri"":null}]",null,dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00004.pdf
"Customer Feedback Report Feedback ID: FB00007 Account ID: ACC00241 Opportunity ID: OPP000289 Feedback Date: 2026-08-14 Sentiment: Neutral Sentiment Score: 0.328 Customer Role: IT Director Source: Survey Feedback Content: Mixed impressions The system appears capable, but implementation may be complex for our team.","{""elements"":[{""bbox"":[{""coord"":[592,125,1108,169],""page_id"":0}],""confidence"":1,""content"":""Customer Feedback Report"",""description"":null,""id"":0,""type"":""title""},{""bbox"":[{""coord"":[197,235,1183,726],""page_id"":0}],""confidence"":1,""content"":""Feedback ID: FB00007\nAccount ID: ACC00241\nOpportunity ID: OPP000289\nFeedback Date: 2026-08-14\nSentiment: Neutral\nSentiment Score: 0.328\nCustomer Role: IT Director\nSource: Survey\n\nFeedback Content:\nMixed impressions\nThe system appears capable, but implementation may be complex for our team."",""description"":null,""id"":1,""type"":""text""}],""pages"":[{""id"":0,""image_uri"":null}]}","[{""id"":0,""image_uri"":null}]",null,dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00007.pdf
"Customer Feedback Report Feedback ID: FB00002 Account ID: ACC00174 Opportunity ID: OPP001959 Feedback Date: 2026-08-08 Sentiment: Positive Sentiment Score: 0.821 Customer Role: Operations Manager Source: Email Feedback Content: Excellent demo! Highlights included real-time reporting, a clean interface, and easy integrations.","{""elements"":[{""bbox"":[{""coord"":[592,125,1108,169],""page_id"":0}],""confidence"":1,""content"":""Customer Feedback Report"",""description"":null,""id"":0,""type"":""title""},{""bbox"":[{""coord"":[197,235,1379,686],""page_id"":0}],""confidence"":1,"

In [0]:
display(customer_feedback_parsed)

path,modificationTime,length,content
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00004.pdf,2026-08-30T14:32:11.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgOCAwIFIgL01lZGlhQm94IFsgMCAwIDYxMiA3OTIgXSAvUGFyZW50IDcgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjUgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyA3IDAgUiAvVHlwZSAvQ2F0YWxvZwo+PgplbmRvYmoKNiAwIG9iago8PAovQXV0aG9yIChhbm9ueW1vdXMpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvQ3JlYXRvciAoYW5vbnltb3VzKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAodW5zcGVjaWZpZWQpIC9UaXRsZSAodW50aXRsZWQpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKNyAwIG9iago8PAovQ291bnQgMSAvS2lkcyBbIDQgMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDQ0NAo+PgpzdHJlYW0KR2FzMkVkO0ZPaSdTYygpTVIwcDNSQi4pUmthZUdbOT1BTXUoWzAjKFlzRS50MSVYZDYjV1YiJTFjM0o3SWZGbkRsMm42LkQuWldGYltQZyVdJjZKQkdNcj1YJitMI1lpXidLKzddNUUhW0hBSG4zVDYlZVtgNmdjTjVlJGE/PVIkT0QsY2NpJ3JZUHRSKFJHLyNvaFlNRD0vUEdKR0IrWT4zP2YjQCxuSUs0U18rYy1hOSpRcXRcIk9jRVlgM1opSjs3UCw9cU07KyhadDhPKDUrWysybENbWlslPTJYIm5oLUgnS0ksIkxMJFYvcE5uIWdITT4saU9naHJOK1c0REdjayhZZ1dhK1dtIi5bXmpfL2VRUD03aWRQZDpLIT1TY2ombWk8IzdjYkNpbTg3cWcuOVhNPFFUUl5tbFMwPE1fSCFGZEt0IT0yaTM9QDBnPGZrRGZxN1wkOHQqPSdySmw+Tk5RJTciNEtuNEdAJWhKci44L1E+cWpTM1IuSW50KSI+b1tzPUVobGlsRjUjS3E+X0wnTmExOjojNSp1YVskIilfSCRPNTVadShTPjxXaWlGIUlDbSQ1bH4+ZW5kc3RyZWFtCmVuZG9iagp4cmVmCjAgOQowMDAwMDAwMDAwIDY1NTM1IGYgCjAwMDAwMDAwNjEgMDAwMDAgbiAKMDAwMDAwMDEwMiAwMDAwMCBuIAowMDAwMDAwMjA5IDAwMDAwIG4gCjAwMDAwMDAzMjEgMDAwMDAgbiAKMDAwMDAwMDUxNCAwMDAwMCBuIAowMDAwMDAwNTgyIDAwMDAwIG4gCjAwMDAwMDA4NDMgMDAwMDAgbiAKMDAwMDAwMDkwMiAwMDAwMCBuIAp0cmFpbGVyCjw8Ci9JRCAKWzwyNmQ3MDlmZmRkNWUwYjBmYTZhNWUzMjE0ZDBkNDVmNz48MjZkNzA5ZmZkZDVlMGIwZmE2YTVlMzIxNGQwZDQ1Zjc+XQolIFJlcG9ydExhYiBnZW5lcmF0ZWQgUERGIGRvY3VtZW50IC0tIGRpZ2VzdCAob3BlbnNvdXJjZSkKCi9JbmZvIDYgMCBSCi9Sb290IDUgMCBSCi9TaXplIDkKPj4Kc3RhcnR4cmVmCjE0MzYKJSVFT0YK
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00009.pdf,2026-08-30T14:32:12.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgOCAwIFIgL01lZGlhQm94IFsgMCAwIDYxMiA3OTIgXSAvUGFyZW50IDcgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjUgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyA3IDAgUiAvVHlwZSAvQ2F0YWxvZwo+PgplbmRvYmoKNiAwIG9iago8PAovQXV0aG9yIChhbm9ueW1vdXMpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA4MzAxNDMyMTIrMDAnMDAnKSAvQ3JlYXRvciAoYW5vbnltb3VzKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA4MzAxNDMyMTIrMDAnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAodW5zcGVjaWZpZWQpIC9UaXRsZSAodW50aXRsZWQpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKNyAwIG9iago8PAovQ291bnQgMSAvS2lkcyBbIDQgMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoID

In [0]:
# Load binary PDF files from a volume
pdf_df = spark.read.format("binaryFile").load(f"/Volumes/{catalog_name}/{schema_name}/workday_unstructure_data/customer_feedback/")

display(pdf_df)

# Parse documents using the Databricks AI function ai_parse_document
parsed_df = pdf_df.withColumn(
    "parsed",
    expr("ai_parse_document(content, map('version', '2.0'))")
)

display(parsed_df)

path,modificationTime,length,content
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00004.pdf,2026-08-30T14:32:11.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgOCAwIFIgL01lZGlhQm94IFsgMCAwIDYxMiA3OTIgXSAvUGFyZW50IDcgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjUgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyA3IDAgUiAvVHlwZSAvQ2F0YWxvZwo+PgplbmRvYmoKNiAwIG9iago8PAovQXV0aG9yIChhbm9ueW1vdXMpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvQ3JlYXRvciAoYW5vbnltb3VzKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAodW5zcGVjaWZpZWQpIC9UaXRsZSAodW50aXRsZWQpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKNyAwIG9iago8PAovQ291bnQgMSAvS2lkcyBbIDQgMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDQ0NAo+PgpzdHJlYW0KR2FzMkVkO0ZPaSdTYygpTVIwcDNSQi4pUmthZUdbOT1BTXUoWzAjKFlzRS50MSVYZDYjV1YiJTFjM0o3SWZGbkRsMm42LkQuWldGYltQZyVdJjZKQkdNcj1YJitMI1lpXidLKzddNUUhW0hBSG4zVDYlZVtgNmdjTjVlJGE/PVIkT0QsY2NpJ3JZUHRSKFJHLyNvaFlNRD0vUEdKR0IrWT4zP2YjQCxuSUs0U18rYy1hOSpRcXRcIk9jRVlgM1opSjs3UCw9cU07KyhadDhPKDUrWysybENbWlslPTJYIm5oLUgnS0ksIkxMJFYvcE5uIWdITT4saU9naHJOK1c0REdjayhZZ1dhK1dtIi5bXmpfL2VRUD03aWRQZDpLIT1TY2ombWk8IzdjYkNpbTg3cWcuOVhNPFFUUl5tbFMwPE1fSCFGZEt0IT0yaTM9QDBnPGZrRGZxN1wkOHQqPSdySmw+Tk5RJTciNEtuNEdAJWhKci44L1E+cWpTM1IuSW50KSI+b1tzPUVobGlsRjUjS3E+X0wnTmExOjojNSp1YVskIilfSCRPNTVadShTPjxXaWlGIUlDbSQ1bH4+ZW5kc3RyZWFtCmVuZG9iagp4cmVmCjAgOQowMDAwMDAwMDAwIDY1NTM1IGYgCjAwMDAwMDAwNjEgMDAwMDAgbiAKMDAwMDAwMDEwMiAwMDAwMCBuIAowMDAwMDAwMjA5IDAwMDAwIG4gCjAwMDAwMDAzMjEgMDAwMDAgbiAKMDAwMDAwMDUxNCAwMDAwMCBuIAowMDAwMDAwNTgyIDAwMDAwIG4gCjAwMDAwMDA4NDMgMDAwMDAgbiAKMDAwMDAwMDkwMiAwMDAwMCBuIAp0cmFpbGVyCjw8Ci9JRCAKWzwyNmQ3MDlmZmRkNWUwYjBmYTZhNWUzMjE0ZDBkNDVmNz48MjZkNzA5ZmZkZDVlMGIwZmE2YTVlMzIxNGQwZDQ1Zjc+XQolIFJlcG9ydExhYiBnZW5lcmF0ZWQgUERGIGRvY3VtZW50IC0tIGRpZ2VzdCAob3BlbnNvdXJjZSkKCi9JbmZvIDYgMCBSCi9Sb290IDUgMCBSCi9TaXplIDkKPj4Kc3RhcnR4cmVmCjE0MzYKJSVFT0YK
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00009.pdf,2026-08-30T14:32:12.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgOCAwIFIgL01lZGlhQm94IFsgMCAwIDYxMiA3OTIgXSAvUGFyZW50IDcgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjUgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyA3IDAgUiAvVHlwZSAvQ2F0YWxvZwo+PgplbmRvYmoKNiAwIG9iago8PAovQXV0aG9yIChhbm9ueW1vdXMpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA4MzAxNDMyMTIrMDAnMDAnKSAvQ3JlYXRvciAoYW5vbnltb3VzKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA4MzAxNDMyMTIrMDAnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAodW5zcGVjaWZpZWQpIC9UaXRsZSAodW50aXRsZWQpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKNyAwIG9iago8PAovQ291bnQgMSAvS2lkcyBbIDQgMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoID

path,modificationTime,length,content,parsed
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00004.pdf,2026-08-30T14:32:11.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVudHMgOCAwIFIgL01lZGlhQm94IFsgMCAwIDYxMiA3OTIgXSAvUGFyZW50IDcgMCBSIC9SZXNvdXJjZXMgPDwKL0ZvbnQgMSAwIFIgL1Byb2NTZXQgWyAvUERGIC9UZXh0IC9JbWFnZUIgL0ltYWdlQyAvSW1hZ2VJIF0KPj4gL1JvdGF0ZSAwIC9UcmFucyA8PAoKPj4gCiAgL1R5cGUgL1BhZ2UKPj4KZW5kb2JqCjUgMCBvYmoKPDwKL1BhZ2VNb2RlIC9Vc2VOb25lIC9QYWdlcyA3IDAgUiAvVHlwZSAvQ2F0YWxvZwo+PgplbmRvYmoKNiAwIG9iago8PAovQXV0aG9yIChhbm9ueW1vdXMpIC9DcmVhdGlvbkRhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvQ3JlYXRvciAoYW5vbnltb3VzKSAvS2V5d29yZHMgKCkgL01vZERhdGUgKEQ6MjAyNjA4MzAxNDMyMTErMDAnMDAnKSAvUHJvZHVjZXIgKFJlcG9ydExhYiBQREYgTGlicmFyeSAtIFwob3BlbnNvdXJjZVwpKSAKICAvU3ViamVjdCAodW5zcGVjaWZpZWQpIC9UaXRsZSAodW50aXRsZWQpIC9UcmFwcGVkIC9GYWxzZQo+PgplbmRvYmoKNyAwIG9iago8PAovQ291bnQgMSAvS2lkcyBbIDQgMCBSIF0gL1R5cGUgL1BhZ2VzCj4+CmVuZG9iago4IDAgb2JqCjw8Ci9GaWx0ZXIgWyAvQVNDSUk4NURlY29kZSAvRmxhdGVEZWNvZGUgXSAvTGVuZ3RoIDQ0NAo+PgpzdHJlYW0KR2FzMkVkO0ZPaSdTYygpTVIwcDNSQi4pUmthZUdbOT1BTXUoWzAjKFlzRS50MSVYZDYjV1YiJTFjM0o3SWZGbkRsMm42LkQuWldGYltQZyVdJjZKQkdNcj1YJitMI1lpXidLKzddNUUhW0hBSG4zVDYlZVtgNmdjTjVlJGE/PVIkT0QsY2NpJ3JZUHRSKFJHLyNvaFlNRD0vUEdKR0IrWT4zP2YjQCxuSUs0U18rYy1hOSpRcXRcIk9jRVlgM1opSjs3UCw9cU07KyhadDhPKDUrWysybENbWlslPTJYIm5oLUgnS0ksIkxMJFYvcE5uIWdITT4saU9naHJOK1c0REdjayhZZ1dhK1dtIi5bXmpfL2VRUD03aWRQZDpLIT1TY2ombWk8IzdjYkNpbTg3cWcuOVhNPFFUUl5tbFMwPE1fSCFGZEt0IT0yaTM9QDBnPGZrRGZxN1wkOHQqPSdySmw+Tk5RJTciNEtuNEdAJWhKci44L1E+cWpTM1IuSW50KSI+b1tzPUVobGlsRjUjS3E+X0wnTmExOjojNSp1YVskIilfSCRPNTVadShTPjxXaWlGIUlDbSQ1bH4+ZW5kc3RyZWFtCmVuZG9iagp4cmVmCjAgOQowMDAwMDAwMDAwIDY1NTM1IGYgCjAwMDAwMDAwNjEgMDAwMDAgbiAKMDAwMDAwMDEwMiAwMDAwMCBuIAowMDAwMDAwMjA5IDAwMDAwIG4gCjAwMDAwMDAzMjEgMDAwMDAgbiAKMDAwMDAwMDUxNCAwMDAwMCBuIAowMDAwMDAwNTgyIDAwMDAwIG4gCjAwMDAwMDA4NDMgMDAwMDAgbiAKMDAwMDAwMDkwMiAwMDAwMCBuIAp0cmFpbGVyCjw8Ci9JRCAKWzwyNmQ3MDlmZmRkNWUwYjBmYTZhNWUzMjE0ZDBkNDVmNz48MjZkNzA5ZmZkZDVlMGIwZmE2YTVlMzIxNGQwZDQ1Zjc+XQolIFJlcG9ydExhYiBnZW5lcmF0ZWQgUERGIGRvY3VtZW50IC0tIGRpZ2VzdCAob3BlbnNvdXJjZSkKCi9JbmZvIDYgMCBSCi9Sb290IDUgMCBSCi9TaXplIDkKPj4Kc3RhcnR4cmVmCjE0MzYKJSVFT0YK,"{""document"":{""elements"":[{""bbox"":[{""coord"":[593,128,1107,169],""page_id"":0}],""confidence"":1,""content"":""Customer Feedback Report"",""description"":null,""id"":0,""type"":""title""},{""bbox"":[{""coord"":[197,235,1379,686],""page_id"":0}],""confidence"":1,""content"":""Feedback ID: FB00004\nAccount ID: ACC00680\nOpportunity ID: OPP001604\nFeedback Date: 2026-08-27\nSentiment: Positive\nSentiment Score: 0.768\nCustomer Role: Operations Manager\nSource: Survey\n\nFeedback Content:\nExcellent demo! Highlights included real-time reporting, a clean interface, and easy integrations."",""description"":null,""id"":1,""type"":""text""}],""pages"":[{""id"":0,""image_uri"":null}]},""error_status"":null,""metadata"":{""file_metadata"":null,""id"":""8db587cb-113a-47a4-89e1-c2d4d82602d6"",""version"":""2.0""}}"
dbfs:/Volumes/demo_rag/workday_demos/workday_unstructure_data/customer_feedback/FB00009.pdf,2026-08-30T14:32:12.000Z,1827,JVBERi0xLjMKJZOMi54gUmVwb3J0TGFiIEdlbmVyYXRlZCBQREYgZG9jdW1lbnQgKG9wZW5zb3VyY2UpCjEgMCBvYmoKPDwKL0YxIDIgMCBSIC9GMiAzIDAgUgo+PgplbmRvYmoKMiAwIG9iago8PAovQmFzZUZvbnQgL0hlbHZldGljYSAvRW5jb2RpbmcgL1dpbkFuc2lFbmNvZGluZyAvTmFtZSAvRjEgL1N1YnR5cGUgL1R5cGUxIC9UeXBlIC9Gb250Cj4+CmVuZG9iagozIDAgb2JqCjw8Ci9CYXNlRm9udCAvSGVsdmV0aWNhLUJvbGQgL0VuY29kaW5nIC9XaW5BbnNpRW5jb2RpbmcgL05hbWUgL0YyIC9TdWJ0eXBlIC9UeXBlMSAvVHlwZSAvRm9udAo+PgplbmRvYmoKNCAwIG9iago8PAovQ29udGVud

In [0]:
customer_feedback_parsed.createOrReplaceTempView("vf_customer_feedback")
meeting_notes_parsed.createOrReplaceTempView("vf_meeting_notes")
email_communications_parsed.createOrReplaceTempView("vf_email_communications")

def create_kb_table_from_parsed(view_name, kb_table_fqn):
    """Create knowledge base table from ai_parse_document output using SQL"""
    # Drop and recreate table
    spark.sql(f"DROP TABLE IF EXISTS {kb_table_fqn}")
    
    spark.sql(f"""
        CREATE TABLE {kb_table_fqn} (
            id BIGINT GENERATED ALWAYS AS IDENTITY,
            content STRING,
            doc_uri STRING
        ) TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """)
    
    # Insert data directly from temp view
    spark.sql(f"""
        INSERT INTO {kb_table_fqn} (content, doc_uri)
        SELECT
            content,
            doc_uri
        FROM {view_name}
        WHERE content IS NOT NULL
    """)
    
    record_count = spark.table(kb_table_fqn).count()
    print(f"✅ {kb_table_fqn} created with {record_count} records")

# Create knowledge base tables from parsed documents
create_kb_table_from_parsed(
    "vf_customer_feedback",
    f"{catalog_name}.{schema_name}.customer_feedback_knowledge_base"
)

create_kb_table_from_parsed(
    "vf_meeting_notes",
    f"{catalog_name}.{schema_name}.meeting_notes_knowledge_base"
)

create_kb_table_from_parsed(
    "vf_email_communications",
    f"{catalog_name}.{schema_name}.email_communications_knowledge_base"
)

✅ andrea_tardif_v2.workday_demos.customer_feedback_knowledge_base created with 25 records
✅ andrea_tardif_v2.workday_demos.meeting_notes_knowledge_base created with 25 records
✅ andrea_tardif_v2.workday_demos.email_communications_knowledge_base created with 25 records


In [0]:
vs_endpoint_name = f"sales-endpoint-{catalog_name}"

# Email communications index
email_vs_index_name = f"{catalog_name}.{schema_name}.email_communications_index"
email_vs_input_table = f"{catalog_name}.{schema_name}.email_communications_knowledge_base"

# Meeting notes index
notes_vs_index_name = f"{catalog_name}.{schema_name}.meeting_notes_index"
notes_vs_input_table = f"{catalog_name}.{schema_name}.meeting_notes_knowledge_base"

# Customer feedback index
feedback_vs_index_name = f"{catalog_name}.{schema_name}.customer_feedback_index"
feedback_vs_input_table = f"{catalog_name}.{schema_name}.customer_feedback_knowledge_base"

In [0]:
from databricks.vector_search.client import VectorSearchClient

# Create vector search endpoint
client = VectorSearchClient(disable_notice=True)

try:
    client.delete_endpoint(vs_endpoint_name)
    print(f"ℹ️  Vector search endpoint '{vs_endpoint_name}' deleted")
    
except Exception as e:
    print(f"ℹ️  Vector search endpoint '{vs_endpoint_name}' did not exist or could not be deleted")

    client.create_endpoint(
        name=vs_endpoint_name,
        endpoint_type="STANDARD"
    )
    print(f"✅ Vector search endpoint '{vs_endpoint_name}' created successfully")

ℹ️  Vector search endpoint 'sales-endpoint-andrea_tardif_v2' did not exist or could not be deleted
✅ Vector search endpoint 'sales-endpoint-andrea_tardif_v2' created successfully


In [0]:
import time

def create_vs_index(endpoint_name, source_table, index_name):
    """Create a vector search index with error handling"""
    try:
        index = client.create_delta_sync_index(
            endpoint_name=endpoint_name,
            source_table_name=source_table,
            index_name=index_name,
            pipeline_type="TRIGGERED",
            primary_key="id",
            embedding_source_column="content",
            embedding_model_endpoint_name="databricks-bge-large-en"
        )
        print(f"✅ {index_name} created successfully")
        return index
    
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"ℹ️  {index_name} already exists")

        else:
            print(f"❌ Error creating {index_name}: {str(e)}")
            return None

# Create all three indexes
email_index = create_vs_index(
    vs_endpoint_name, 
    email_vs_input_table, 
    email_vs_index_name,
)

notes_index = create_vs_index(
    vs_endpoint_name, 
    notes_vs_input_table, 
    notes_vs_index_name,
)

feedback_index = create_vs_index(
    vs_endpoint_name, 
    feedback_vs_input_table, 
    feedback_vs_index_name,
)

✅ andrea_tardif_v2.workday_demos.email_communications_index created successfully
✅ andrea_tardif_v2.workday_demos.meeting_notes_index created successfully
✅ andrea_tardif_v2.workday_demos.customer_feedback_index created successfully


In [0]:
print("Syncing vector search indexes...")

for index_name in [
    (email_vs_index_name),
    (notes_vs_index_name),
    (feedback_vs_index_name)
]:
    try:
        client.get_index(endpoint_name=vs_endpoint_name, index_name=index_name).sync()
        print(f"✅ {index_name} index synced")
    except Exception as e:
        print(f"⚠️  Could not sync {index_name} index: {str(e)}")

print("\n🎉 Vector search setup complete!")

Syncing vector search indexes...
⚠️  Could not sync andrea_tardif_v2.workday_demos.email_communications_index index: Response content b'{"error_code":"BAD_REQUEST","message":"Vector search endpoint sales-endpoint-andrea_tardif_v2 is not ready yet.","details":[{"@type":"type.googleapis.com/google.rpc.RequestInfo","request_id":"0f5173aa-fd15-4e6e-93f0-0f846ed4ee06","serving_data":""}]}', status_code 400
⚠️  Could not sync andrea_tardif_v2.workday_demos.meeting_notes_index index: Response content b'{"error_code":"BAD_REQUEST","message":"Vector search endpoint sales-endpoint-andrea_tardif_v2 is not ready yet.","details":[{"@type":"type.googleapis.com/google.rpc.RequestInfo","request_id":"9013cc41-d58b-4658-82df-10d7a1f2f239","serving_data":""}]}', status_code 400
⚠️  Could not sync andrea_tardif_v2.workday_demos.customer_feedback_index index: Response content b'{"error_code":"BAD_REQUEST","message":"Vector search endpoint sales-endpoint-andrea_tardif_v2 is not ready yet.","details":[{"@typ